### Machine Learning | Assignment 2 - Supervised Learning | Exercise 4

145873 | Sezer Tetik


In [7]:
import numpy as np
import pandas as pd
import math

In [8]:
iris = pd.read_csv("iris.csv")

iris

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [9]:
iris_specs = iris.iloc[:, :4].to_numpy()
iris_species = (iris["species"] == 0).astype(int).to_numpy()

cuts = np.quantile(iris_specs, [1/3, 2/3], axis=0)

labels = np.array(["low", "medium", "high"])
categorized = np.column_stack([
    labels[np.digitize(iris_specs[:, i], cuts[:, i])]
    for i in range(iris_specs.shape[1])
])

features = np.empty(iris_specs.shape, dtype=object) # Columns 0-3 hold (value, label) pairs, column 4 the target (1 = Iris-setosa)
for i, j in np.ndindex(features.shape):
    features[i, j] = (float(iris_specs[i, j]), str(categorized[i, j]))

dataset = np.column_stack((features, iris_species))
datasets = {
    category: dataset[[label == category for _, label in dataset[:, 0]]]
    for category in ["low", "medium", "high"]
}
datasets["all"] = dataset

datasets["low"]

array([[(5.1, 'low'), (3.5, 'high'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.9, 'low'), (3.0, 'medium'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.7, 'low'), (3.2, 'high'), (1.3, 'low'), (0.2, 'low'), 1],
       [(4.6, 'low'), (3.1, 'medium'), (1.5, 'low'), (0.2, 'low'), 1],
       [(5.0, 'low'), (3.6, 'high'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.6, 'low'), (3.4, 'high'), (1.4, 'low'), (0.3, 'low'), 1],
       [(5.0, 'low'), (3.4, 'high'), (1.5, 'low'), (0.2, 'low'), 1],
       [(4.4, 'low'), (2.9, 'medium'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.9, 'low'), (3.1, 'medium'), (1.5, 'low'), (0.1, 'low'), 1],
       [(4.8, 'low'), (3.4, 'high'), (1.6, 'low'), (0.2, 'low'), 1],
       [(4.8, 'low'), (3.0, 'medium'), (1.4, 'low'), (0.1, 'low'), 1],
       [(4.3, 'low'), (3.0, 'medium'), (1.1, 'low'), (0.1, 'low'), 1],
       [(5.1, 'low'), (3.5, 'high'), (1.4, 'low'), (0.3, 'low'), 1],
       [(5.1, 'low'), (3.8, 'high'), (1.5, 'low'), (0.3, 'low'), 1],
       [(5.1, 'low'), 

Each column is categorized with the same tertile cuts as in the previous exercise (low / medium / high), and the target is 1 for Iris-setosa and 0 for the remaining species. The three new datasets are created by grouping the examples by the category of the first column (sepal length), whose cuts are 5.4 and 6.3.

The datasets do not have 50 examples each, because they group the flowers by sepal length range, not by species. The table below shows how the two groupings overlap:

| Sepal length | Setosa | Versicolor | Virginica | Total |
|---|---:|---:|---:|---:|
| low (< 5.4) | 40 | 5 | 1 | 46 |
| medium (5.4 – 6.3) | 10 | 31 | 12 | 53 |
| high (≥ 6.3) | 0 | 14 | 37 | 51 |
| **Total** | **50** | **50** | **50** | **150** |

Each species still has 50 examples, but sepal length does not separate them perfectly, so each subset contains more than one species. The ranges are also not exactly one third of the data each: 6 flowers have a sepal length of exactly 5.4 and 9 have exactly 6.3, and `np.digitize` places a value equal to a cut in the upper category. Forcing exactly 50 examples per subset would put flowers with the same sepal length into different categories.

In [10]:
def calculate_entropy(dataset):
    labels = dataset[:, 4]

    p_plus = len(dataset[labels == 1]) / len(labels)
    p_negative = len(dataset[labels == 0]) / len(labels)

    entropy = 0
    if p_plus > 0:
        entropy -= p_plus * math.log2(p_plus)

    if p_negative > 0:
        entropy -= p_negative * math.log2(p_negative)

    return entropy

def calculate_gain(dataset: np.ndarray, feature: int):
    entropy = calculate_entropy(dataset)
    weighted_entropy = 0

    categories = np.array([item[1] for item in dataset[:, feature]]) # Extract categorical labels for the selected feature
    for category in ["low", "medium", "high"]:
        subset = dataset[categories == category]

        if len(subset) == 0:
            continue

        weighted_entropy += (
            len(subset) / len(dataset)
        ) * calculate_entropy(subset)

    return entropy - weighted_entropy

In [11]:
for category, data in datasets.items():
    setosa = int(data[:, 4].sum())
    print(
        f"{category}:\n", 
        f"setosa: {setosa}/{len(data)} ({setosa / len(data):.2f})\n",
        f"entropy: {calculate_entropy(data):.2f}\n", 
        f"gains:\n",
        f"  feature (sepal_length): {calculate_gain(data, 0):.2f}\n",
        f"  feature (sepal_width): {calculate_gain(data, 1):.2f}\n",
        f"  feature (petal_length): {calculate_gain(data, 2):.2f}\n",
        f"  feature (petal_width): {calculate_gain(data, 3):.2f}\n",
        end="\n")

low:
 setosa: 40/46 (0.87)
 entropy: 0.56
 gains:
   feature (sepal_length): 0.00
   feature (sepal_width): 0.47
   feature (petal_length): 0.56
   feature (petal_width): 0.56

medium:
 setosa: 10/53 (0.19)
 entropy: 0.70
 gains:
   feature (sepal_length): 0.00
   feature (sepal_width): 0.51
   feature (petal_length): 0.70
   feature (petal_width): 0.70

high:
 setosa: 0/51 (0.00)
 entropy: 0.00
 gains:
   feature (sepal_length): 0.00
   feature (sepal_width): 0.00
   feature (petal_length): 0.00
   feature (petal_width): 0.00

all:
 setosa: 50/150 (0.33)
 entropy: 0.92
 gains:
   feature (sepal_length): 0.50
   feature (sepal_width): 0.29
   feature (petal_length): 0.92
   feature (petal_width): 0.92



This output answers question 1 (entropy of the 4 datasets) and, in the "all" block, gives the gains of every feature on the complete dataset used in questions 2 and 3.

In [12]:
def explain_gain(dataset: np.ndarray, feature: int, name: str):
    entropy = calculate_entropy(dataset)
    gain = calculate_gain(dataset, feature)
    print(f"Entropy(S) = {entropy:.2f}\n")

    weighted = []
    categories = np.array([item[1] for item in dataset[:, feature]]) # Extract categorical labels for the selected feature
    for category in ["low", "medium", "high"]:
        subset = dataset[categories == category]

        if len(subset) == 0:
            continue

        setosa = int(subset[:, 4].sum())
        p_plus = setosa / len(subset)
        p_negative = (len(subset) - setosa) / len(subset)
        subset_entropy = calculate_entropy(subset)
        weighted.append(f"{len(subset)} x {subset_entropy:.2f} / {len(dataset)}")

        terms = " - ".join(f"{p:.2f} x log2({p:.2f})" for p in (p_plus, p_negative) if p > 0) # Same as calculate_entropy: p = 0 terms are skipped

        print(f"{name} = {category}")
        print(f"  P(setosa | {name} = {category}) = {setosa}/{len(subset)} = {p_plus:.2f}")
        print(f"  P(not setosa | {name} = {category}) = {len(subset) - setosa}/{len(subset)} = {p_negative:.2f}")
        print(f"  Entropy = -{terms} = {subset_entropy:.2f}\n")

    print(f"gain(S, {name}) = {entropy:.2f} - ({' + '.join(weighted)}) = {entropy:.2f} - {entropy - gain:.2f} = {gain:.2f}")

for feature, name in enumerate(["sepal_length", "sepal_width", "petal_length", "petal_width"]):
    print(f"--- {name} ---")
    explain_gain(datasets["all"], feature, name)
    print()

--- sepal_length ---
Entropy(S) = 0.92

sepal_length = low
  P(setosa | sepal_length = low) = 40/46 = 0.87
  P(not setosa | sepal_length = low) = 6/46 = 0.13
  Entropy = -0.87 x log2(0.87) - 0.13 x log2(0.13) = 0.56

sepal_length = medium
  P(setosa | sepal_length = medium) = 10/53 = 0.19
  P(not setosa | sepal_length = medium) = 43/53 = 0.81
  Entropy = -0.19 x log2(0.19) - 0.81 x log2(0.81) = 0.70

sepal_length = high
  P(setosa | sepal_length = high) = 0/51 = 0.00
  P(not setosa | sepal_length = high) = 51/51 = 1.00
  Entropy = -1.00 x log2(1.00) = 0.00

gain(S, sepal_length) = 0.92 - (46 x 0.56 / 150 + 53 x 0.70 / 150 + 51 x 0.00 / 150) = 0.92 - 0.42 = 0.50

--- sepal_width ---
Entropy(S) = 0.92

sepal_width = low
  P(setosa | sepal_width = low) = 1/47 = 0.02
  P(not setosa | sepal_width = low) = 46/47 = 0.98
  Entropy = -0.02 x log2(0.02) - 0.98 x log2(0.98) = 0.15

sepal_width = medium
  P(setosa | sepal_width = medium) = 11/47 = 0.23
  P(not setosa | sepal_width = medium) = 36/4

**1. Entropy of the 4 datasets**

Entropy measures how uncertain we are about the class of an example drawn from a set. It is 0 when the set is pure (all setosa or all non-setosa) and 1 when the two classes are evenly mixed (50% / 50%).

**All: ~0.92** - 50 of 150 examples are setosa (33%). The classes are fairly mixed, so before any split it is hard to guess the class of a random example.

**Low: ~0.56** - 40 of 46 examples are setosa (87%). The set is mostly setosa, so uncertainty is moderate: guessing "setosa" would be right most of the time, but not always.

**Medium: ~0.70** - 10 of 53 examples are setosa (19%). The set is mostly non-setosa but more mixed than the low set, so it has the highest uncertainty of the three subsets.

**High: 0.00** - 0 of 51 examples are setosa. The set is pure, so there is no uncertainty: every example is non-setosa.

| Metric | Low | Medium | High | All |
|---|---:|---:|---:|---:|
| **Setosa / Total** | 40 / 46 | 10 / 53 | 0 / 51 | 50 / 150 |
| **Setosa (%)** | 86.96% | 18.87% | 0.00% | 33.33% |
| **Entropy** | 0.56 | 0.70 | 0.00 | 0.92 |
| **Gain: Sepal Length** | 0.00 | 0.00 | 0.00 | 0.50 |
| **Gain: Sepal Width** | 0.47 | 0.51 | 0.00 | 0.29 |
| **Gain: Petal Length** | **0.56** | **0.70** | 0.00 | **0.92** |
| **Gain: Petal Width** | **0.56** | **0.70** | 0.00 | **0.92** |

**2. Gain of the split by the first column (sepal length)**

$$gain(S, sepal\_length) = 0.92 - \left(\frac{46 \times 0.56}{150} + \frac{53 \times 0.70}{150} + \frac{51 \times 0.00}{150}\right) = 0.92 - 0.42 = 0.50$$

Before the split, the uncertainty about the class of an example is 0.92 bits. After splitting by sepal length, the weighted average uncertainty of the three subsets is 0.42 bits, so the split removes 0.50 bits, a bit more than half of the original uncertainty. In practice, after the split we know for sure that an example with high sepal length is not setosa, and in the low and medium subsets one class is clearly in the majority (87% setosa and 81% non-setosa). The split does not classify every example correctly, though: the low and medium subsets still contain both classes.

The sepal length gain is 0 inside the low, medium and high subsets because each subset already has a single sepal length value, so splitting it again by that feature changes nothing.

**3. Gain of all features**

On the complete dataset, petal length and petal width have the greatest gain (0.92), followed by sepal length (0.50) and sepal width (0.29). The gain of petal length and petal width equals the entropy of the whole dataset, so the entropy left after the split is 0: every subset (low, medium, high) is pure. All 50 setosa examples have low petal length and low petal width, and no non-setosa example does.

This tells us which feature to look at first to improve our chances of guessing the class of a random example. Without any information, the best guess is always "non-setosa", which is right 67% of the time. If we know the example's petal length (or petal width), we can guess "setosa" when it is low and "non-setosa" otherwise, and we are right 100% of the time on this dataset. Sepal length alone helps too (a majority-class guess in each subset is right (40 + 43 + 51) / 150 ≈ 89% of the time), but less.

**4. Building a decision tree**

A decision tree (ID3 algorithm) can be built from these gains as follows:

1. Calculate the gain of every feature on the current set and choose the feature with the greatest gain as the node.
2. Create one branch for each value of that feature (low, medium, high) and send each example down the branch that matches its value.
3. For each branch:
   - If the subset is pure (entropy = 0), make it a leaf labeled with that class.
   - If no features are left, make it a leaf labeled with the majority class.
   - Otherwise, repeat from step 1 on the subset, using only the features that have not been used yet on that path.

On our dataset, petal length has the greatest gain (0.92, tied with petal width; we pick petal length), so it becomes the root. All three subsets it creates are pure, so the tree stops after one split:

```
            petal_length
          /      |       \
       low    medium     high
        |        |         |
     setosa  not setosa  not setosa
```

Had we used sepal length as the root instead (gain 0.50), the high branch would be a pure leaf (not setosa), but the low and medium branches would still be mixed. Inside those branches, petal length and petal width would again have the greatest gain (0.56 for low and 0.70 for medium, equal to the subsets' entropy), so a second level of splits would be needed to separate the classes. Choosing the feature with the greatest gain at each node is what keeps the tree small.